# Probability Current Decomposition

Chapter 21: Fokker-Planck and Distribution Dynamics · *Mathematics for AI and Machine Learning* by Fuheng Wu

Visualize drift and diffusion contributions to Fokker-Planck probability current.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-21/probability-current/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

# The demo's defaults: drift f(x) = -kappa x, diffusion D, density p = N(mu, sig^2)
kappa, D, mu, sig = 1.1, 0.35, 0.8, 0.7
x = torch.linspace(-6, 6, 12001, dtype=torch.float64)
p = torch.exp(-0.5 * ((x - mu) / sig) ** 2) / (math.sqrt(2 * math.pi) * sig)
dp = -(x - mu) / sig ** 2 * p

# ── Probability current J = f p - D dp/dx (Fokker-Planck: dp/dt = -dJ/dx) ──
J = -kappa * x * p - D * dp
dpdt = -torch.gradient(J, spacing=(x,))[0]
print(f"max |J| = {J.abs().max().item():.4f} at x = {x[J.abs().argmax()].item():+.3f}")
print(f"total mass change d/dt ∫p = {torch.trapezoid(dpdt, x).item():+.2e} (probability is conserved)")
# This p is not stationary: the drift pulls the mean toward 0 at rate kappa
print(f"d/dt of the mean = ∫ x dp/dt = {torch.trapezoid(x * dpdt, x).item():+.4f}  (theory -kappa mu = {-kappa * mu:+.4f})")

# ── The stationary density has zero current everywhere ──────────────────────
s2 = D / kappa                                        # N(0, D / kappa)
ps = torch.exp(-0.5 * x ** 2 / s2) / math.sqrt(2 * math.pi * s2)
Js = -kappa * x * ps - D * torch.gradient(ps, spacing=(x,))[0]
print(f"\nstationary N(0, D/kappa = {s2:.4f}): max |J| = {Js.abs().max().item():.2e}  (drift and diffusion balance)")

max |J| = 0.5237 at x = +0.998
total mass change d/dt ∫p = +1.71e-12 (probability is conserved)
d/dt of the mean = ∫ x dp/dt = -0.8800  (theory -kappa mu = -0.8800)

stationary N(0, D/kappa = 0.3182): max |J| = 3.17e-07  (drift and diffusion balance)
